In [ ]:
from pathlib import Path
import csv, json
from typing import Dict, List, Any, Tuple

WORKSPACE = Path(__file__).resolve().parents[1]
PILOT_DIR = WORKSPACE / 'outputs' / 'pilot'
STRUCT_REFS_DIR = WORKSPACE / 'outputs' / 'structured_refs'


def read_csv(path: Path) -> List[Dict[str, str]]:
    rows: List[Dict[str, str]] = []
    with open(path, 'r', encoding='utf-8') as f:
        r = csv.DictReader(f)
        for row in r:
            rows.append(row)
    return rows


def load_refs() -> Dict[Tuple[str, Any], List[Dict[str, Any]]]:
    m: Dict[Tuple[str, Any], List[Dict[str, Any]]] = {}
    for fn in ['dem_claim_references.json', 'rep_claim_references.json']:
        p = STRUCT_REFS_DIR / fn
        if not p.exists():
            continue
        with open(p, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for row in data:
            pr = str(row.get('press_release'))
            cn = row.get('claim_number')
            m[(pr, cn)] = row.get('references') or []
    return m


def show_claim_with_refs(press_release: str, claim_number: Any, k: int = 5) -> None:
    pairs = read_csv(PILOT_DIR / 'pilot_pairs_assignments.csv')
    refs_map = load_refs()
    # find the claim text
    claim_text = None
    for r in pairs:
        if r.get('press_release') == press_release and str(r.get('claim_number')) == str(claim_number):
            claim_text = r.get('claim_text')
            break
    print('--- Claim ---')
    print(f'{press_release} / #{claim_number}')
    print(claim_text or '[not found]')
    print('\n--- References ---')
    refs = refs_map.get((str(press_release), claim_number), [])
    for i, ref in enumerate(refs[:k], start=1):
        title = ref.get('title')
        venue = ref.get('venue')
        year = ref.get('year')
        doi = ref.get('doi')
        url = ref.get('url')
        print(f'[{i}] {title} ({venue}, {year})')
        print(f'    DOI: {doi}  URL: {url}')

